<a href="https://colab.research.google.com/github/jamg-upv/LLMforSLRscreening/blob/main/py/API_PROMPT_ART_749_screeningSLR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Descripción del caso de uso
basado en chatgptAPI_PROMPT_UBschoolMay25.ipynb
Original: Reto21-chatgptAPI-CasoUso1HIWPclasificacion.ipynb

A partir del bot que tengo en poe.com sobre entity extraction y clasificación de resumenes de artículos: Bot04clasificHIWP

LA entrada es una tabla con al menos tres columnas
Id |	Title |	Abstract
El resultado quiero que me genere una tabla con
Id#Entity extraction separada por ";"#clasificacion#rationale para la clasificación

Manualmente copiando y pegando el input y los resultados (de 20 en 20) de un chat de IAgen me funciona dentro de la ventana de contexto. Pero quiero evitar tener que hacerlo a mano.

# Archivo de trabajo. Siempre hay que leer uno al principio de la sesión

In [ ]:
# En Python, puedes usar comillas simples o dobles para definir una cadena de texto (string). Ambas son válidas y no hay diferencia funcional entre ellas.
#  si tu cadena de texto contiene una comilla simple (como en "I'm"), puedes usar comillas dobles para evitar tener que escapar la comilla simple. De manera similar, si tu cadena de texto contiene una comilla doble, puedes usar comillas
import os
# RutaData = input("Ruta completa al archivo (ejemplo: https://gdu.uv.es/sh/alfresco.xls ): ")
# "ART-749-SCAg-WOS-SCOListaV2data4screeningShort20pilot.xlsx"
# "ART-749-SCAg-WOS-SCOListaSinDuplicadosV2data4screeningFull593.xlsx"
RutaBase="/content/"
FileName="ART-749-SCAg-WOS-SCOListaV2data4screeningShort20pilot.xlsx"
RutaData= os.path.join(RutaBase,FileName)

# Este código lee el archivo XLS ubicado en la URL especificada y lo carga en un DataFrame de pandas llamado DataPapers.
# Debes tener en cuenta que read_excel() necesita la biblioteca openpyxl o xlrd para leer archivos de Excel,
# si no la tienes instalada, puedes hacerlo con pip install openpyxl o pip install xlrd.
# Además, también necesitas tener la biblioteca requests para leer archivos desde una URL. Si no la tienes instalada, puedes hacerlo con pip install requests.

import pandas as pd

# Leemos el archivo en un DataFrame
DataPapers = pd.read_excel(RutaData)

DataPapers['Prompt'] = DataPapers['Id'].astype(str) + "# "  + DataPapers['Title'] + ". "+ DataPapers['Abstract']
DataPapers

# Definimos las columnas que queremos mantener
columnas = ["Id", "Title","Abstract", "Prompt"]

# Seleccionamos las columnas y las primeras 8 filas
# DataPapers = DataPapers[columnas].iloc[:8]

# Seleccionamos las columnas y TODAS las filas (no solo las primeras 10)
DataPapers = DataPapers[columnas]  # Eliminamos el .iloc[:10]


DataPapers



# CArgar promt. Siempre hay que correr una vez (al menos) al principio de sesion

In [ ]:
PromtSystemA = '''
System / Role Instruction (Context for the AI Model)
You are an operations management researcher interested in screening scientific articles related to:
“Research that empirically analyzes how a supply chain’s ability to detect and respond quickly to unexpected changes in supply and demand affects business performance. Supply Chain Agility is a multidimensional organizational capability that enables rapid detection and response to unexpected changes in supply and demand through some of these elements: (1) accessibility and agile acquisition of relevant data, (2) early alertness to threats and opportunities, (3) effective decision-making based on available information, (4) flexibility to adapt operations and tactics, (5) joint planning among supply chain actors, (6) integration of collaborative processes between buyers and suppliers, and (7) the ability to make and rapidly implement decisions on supply chain and logistics management. This capability enables maintaining or generating competitive advantage through timely proactive or reactive responses that meet changing market requirements. The study must examine how these capabilities impact organizational performance or competitive advantage.”

I will provide a list of abstracts in English, each identified by a unique ID number. For each abstract, please follow these steps:
1.	Output Header
o	Write “##” followed by the abstract’s identifier, then a “#” symbol.
o	Example: ##23JSJZRE--10.17270/J.LOG.2022.735#
2.	Entity Extraction
o	Identify and list the main entities (keywords, concepts, or named entities) you see in the abstract.
o	Separate each extracted entity with a semicolon (;).
o	End this list of entities with a “#” symbol.
o	Example: Supply Chain Agility; Performance; Case Study#
3.	Classification
o	Compare the extracted entities (and any other relevant context in the abstract) with the definition of the topic given above (i.e., “Research that empirically analyzes how a supply chain’s ability to detect and respond quickly to unexpected changes in supply and demand affects business performance...”).
o	Assign one of the following categories to the abstract, followed immediately by a “#” symbol:
	@Cat1InsufficInformat@ if the abstract does not provide enough information to determine relevance.
	@Cat2Sele@ if the abstract clearly addresses the topic of interest (empirical research on supply chain agility and performance).
	@Cat3Maybe@ if it is probable that the abstract deals with the topic but it is not entirely clear.
	@CatDiscard@ in all other cases (the abstract does not address the topic of interest).
4.	Explanation
o	Briefly explain why you classified the abstract in that category.
o	End your explanation with a double ##.
Final Output Format
•	Present the results for all processed abstracts as a table in English with three columns (separated by \t for tabs, and \n for new lines). Do not add an introduction, headings, or extra information before or after the rows
•	Each row corresponds to one abstract.
•	First column: the abstract identifier.
•	Second column: the list of extracted entities (separated by ;).
•	Third column: the assigned category.
Example of the Output (shown here as raw text, so it preserves alignment in Excel):
##23JSJZRE--10.17270/J.LOG.2022.735#Supply Chain Agility; Performance; Case Study#    @Cat2Sele@
##24US6F9Q--10.31838/srp.2020.2.107# ...    @CatDiscard@
Show all the results, respect the format I indicated for field separation, and do not make up blank rows between data
'''

PromtSystemB_old = '''
# Role and Context
You are an operations management scientist specializing in screening scientific articles related to supply chain agility and performance. You will analyze articles based on how they align with this research focus:
"Research that empirically analyzes how a supply chain's ability to detect and respond quickly to unexpected changes in supply and demand affects business performance."

# Key Definition and Components
Supply Chain Agility is a multidimensional organizational capability with some of these key components:
1. Accessibility and agile acquisition of relevant data
2. Early alertness to threats and opportunities
3. Effective decision-making based on available information
4. Flexibility to adapt operations and tactics
5. Joint planning among supply chain actors
6. Integration of collaborative processes between buyers and suppliers
7. Ability to make and rapidly implement decisions on SCM and logistics management

This capability enables maintaining or generating competitive advantage through timely proactive or reactive responses to changing market requirements.

# Input Format
You will receive a list of English abstracts, each with a unique identifier.

# Processing Instructions
For each abstract, follow these four steps:
1. Identifier Marking
- Start with "##" followed by the abstract's identifier and "#"
2. Entity Extraction
- Extract relevant entities from the abstract
- Separate entities with semicolons (;)
- End with "#"
- Focus on extracting:
  * Research methods used
  * Supply chain aspects studied
  * Performance measures
  * Analysis techniques
  * Industry context
  * Key variables
3. Classification
Compare extracted entities with the topic definition and classify into one of these categories:
- @Cat1InsufficInformat@ : Insufficient information for classification
- @Cat2Sele@ : Definitely addresses the research topic
- @Cat3Maybe@ : Probably addresses the topic but unclear
- @Cat4Discard@ : Does not address the research topic
End with "#"
4. Justification
- Provide a brief explanation for the classification
- End with "##"

# Output Format
Present results in a three-column table using:
- Tab-separated columns (\t)
- New lines between rows (\n)
- Columns:
  1. Abstract identifier
  2. Extracted entities (semicolon-separated)
  3. Classification category

Example:
```
Identifier\tExtracted Entities\tClassification\n
ABC123\tSupply chain agility; empirical study; performance measurement; manufacturing sector; regression analysis\t@Cat2Sele@\n
```
# Classification Criteria
To be classified as @Cat2Sele@, abstract must show:
1. Empirical research approach
2. Clear focus on supply chain agility (per definition)
3. Analysis of business performance impact
4. Evidence-based methodology

# Sample Analysis
Input:
```
ID: [ABC123]   Abstract: [Abstract text]
```
Output:
```
##ABC123#Supply chain flexibility; manufacturing performance; empirical survey; structural equation modeling; automotive industry#@Cat2Sele@#The study employs empirical methods to directly examine supply chain agility's impact on performance, using validated measures and appropriate analytical techniques##
```
'''

PromtSystemB = '''
# Role and Context
You are an operations management scientist specializing in screening scientific articles related to supply chain agility and performance. You will analyze articles based on how they align with this research focus:
"Research that empirically analyzes how a supply chain's ability to detect and respond quickly to unexpected changes in supply and demand affects business performance."

# Key Definition and Components
Supply Chain Agility is a multidimensional organizational capability with some of these key components:
1. Accessibility and agile acquisition of relevant data
2. Early alertness to threats and opportunities
3. Effective decision-making based on available information
4. Flexibility to adapt operations and tactics
5. Joint planning among supply chain actors
6. Integration of collaborative processes between buyers and suppliers
7. Ability to make and rapidly implement decisions on SCM and logistics management

This capability enables maintaining or generating competitive advantage through timely proactive or reactive responses to changing market requirements.

# Sample Analysis
Input:
```
ID: [ABC123]   Abstract: [Abstract text]
```
Output:
```
##ABC123#Supply chain flexibility; manufacturing performance; empirical survey; structural equation modeling; automotive industry#@Cat2Sele@#The study employs empirical methods to directly examine supply chain agility's impact on performance, using validated measures and appropriate analytical techniques##
```

# Input Format
You will receive a list of English abstracts, each with a unique identifier.

# Processing Instructions
For each abstract, follow these four steps (respect the format, do not add step names or procedures, and do not make up blank rows between result of each step):
1. Identifier Marking
- Start with "##" followed by the abstract's identifier and "#"
2. Entity Extraction
- Extract relevant entities from the abstract
- Separate entities with semicolons (;)
- End with "#"
- Focus on extracting:
  * Research methods used
  * Supply chain aspects studied
  * Performance measures
  * Analysis techniques
  * Industry context
  * Key variables
3. Classification
Compare extracted entities with the topic definition and classify into one of these categories:
- @Cat1InsufficInformat@ : Insufficient information for classification
- @Cat2Sele@ : Definitely addresses the research topic
- @Cat3Maybe@ : Probably addresses the topic but unclear
- @Cat4Discard@ : Does not address the research topic
End with "#"
4. Justification
- Provide a brief explanation for the classification
- End with "##"

# Classification Criteria
To be classified as @Cat2Sele@, abstract must show:
1. Empirical research approach
2. Clear focus on supply chain agility (per definition)
3. Analysis of business performance impact
4. Evidence-based methodology
'''

PromtSystemC = '''

'''




In [ ]:
# Crear un DataFrame vacío
df3 = pd.DataFrame(columns=['GPTId', 'GPTContent','GPTResponse','GPTUsage','PromtAgregado'])

# Codigo especifico para OPEN AI API




##introducir api key OpenAI (jecutar una vez cada sesión)

In [ ]:
#manual
import getpass

# Solicitar la entrada al usuario
keyOpenAi= getpass.getpass("Introduce API KEY sin comillas: ")

In [ ]:
#automatico

from google.colab import userdata
keyOpenAi = userdata.get('KeyOpenAi')

In [ ]:
!pip install openai

import os
from openai import OpenAI

# Inicialización del cliente con la API moderna
client = OpenAI(api_key=keyOpenAi)

# Selección del modelo precio  Input	Cached input	Output  https://platform.openai.com/docs/pricing
# ModelUsed="gpt-4.1-2025-04-14"        #$2.00 $0.50  S8.00
# ModelUsed="gpt-4o-mini-2024-07-18"  #$0.15 $0.075 $0.60
# ModelUsed="o4-mini-2025-04-16"      #$1.10 $0.275 $4.40
# ModelUsed="gpt-5-2025-08-07"          # 1.25  00125 10.00
ModelUsed="gpt-5-mini-2025-08-07"   #0.25  0.025 2.00
# ModelUsed="gpt-5-nano-2025-08-07"   #0.05 0.005 0.40


## codigo OpenAI

In [ ]:
# código refacturizado: para hacer esto
#  extrayendo el texto de una fila de un dataframe,
#   usándolo como entrada para la API de ChatCompletion de OpenAI,
#   e insertando la respuesta en otro dataframe.
# Este código asume que tienes una cierta flexibilidad en la elección de tus filas de inicio y fin.
#  Si siempre estás procesando la siguiente fila y no tienes un rango fijo,
#   podrías considerar almacenar la "fila actual" en una variable persistente o en un archivo, y actualizarla cada vez que ejecutes la función.

# La función generate_response_and_update_df toma el DataFrame donde quieres guardar la respuesta,
#  el índice de la fila a actualizar, el modelo de OpenAI a utilizar, los mensajes del sistema y del usuario, y la temperatura para la generación de texto.
#  Luego realiza la llamada a la API de OpenAI, imprime la información de uso y el contenido de la respuesta, actualiza el DataFrame con la información de la respuesta y finalmente devuelve el DataFrame actualizado.
import time

def generate_response_and_update_df(df, row, model, system_prompt, user_prompt, temperature):
    # Llamada a la API de OpenAI con la sintaxis actualizada
    response = client.chat.completions.create(
      model=model,
      messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
      temperature=temperature
    )

    # Imprimir la información de los tokens y el contenido de respuesta
    print(response.usage)
    print(response.choices[0].message.content)

    # Añadir la respuesta GPT al DataFrame
    df.loc[row, 'GPTId'] = row
    df.loc[row, 'GPTContent'] = response.choices[0].message.content
    df.loc[row, 'GPTResponse'] = str(response)
    df.loc[row, 'GPTUsage'] = str(response.usage)
    df.loc[row, 'PromtAgregado'] = user_prompt

    return df

# Llamada a la función en un bucle
start_row = 0
end_row = 593  # Debe ser exactamente el numero de filas (porqu este no se ejecuta, pero al empezar en 0 la fila anterior es el total). si es mas el proceso se corta y no ejecutar la descarga de ficheros
fixed_delay = 0.05  # la idea es no pasarse de 200 peticiones por minuto, incluso sin delay va a unas 30 peticiones por minuto

for row in range(start_row, end_row):
    PromtUser1 = DataPapers['Prompt'].iloc[row]
    df3 = generate_response_and_update_df(df3, row, ModelUsed, PromtSystemB, PromtUser1, Temperatura)
    print(f"Procesada fila {row + 1} de {end_row}")

    # Guardar periódicamente para no perder progreso
    if row % 50 == 0 and row > 0:  # Cada 50 filas
        df3.to_csv(f"resultados_parciales_hasta_fila_{row}.csv", index=False)
        print(f"Progreso guardado hasta la fila {row}")

    # Pausa fija entre peticiones
    if row < end_row - 1:  # No esperar después de la última fila
        time.sleep(fixed_delay)

# Guardar resultado final
df3.to_csv("resultados_finales.csv", index=False)
print("Proceso completado!")

df3

# Guardamos el DataFrame en un archivo Excel
from google.colab import files

# Definimos la ruta y el nombre del archivo incluyendo la fecha actual
from datetime import datetime
fecha_actual = datetime.now().strftime("%Y-%m-%d %H:%M")
# nombre_base = f"ART-749-SCAg-api-poe-{ModelUsed}-B01-{Temperatura}"
nombre_base = f"ART-749-SCAg-api-OpenAi-{ModelUsed}-B01-{Temperatura}"
nombre_fichero = f"{nombre_base}_{fecha_actual}.xlsx"

# Guardamos el archivo
df3.to_excel(nombre_fichero, index=False)

# Verificamos que el archivo existe y lo descargamos
if os.path.exists(nombre_fichero):
    files.download(nombre_fichero)
    print(f"Archivo {nombre_fichero} descargado exitosamente")
else:
    print("Error: No se pudo crear el archivo")



# Codigo especifico para GEMINI/Google API

In [ ]:

# Para Gemini
!pip install google-generativeai

import time
import google.generativeai as genai
from google.colab import userdata

# Configurar la API de Gemini
genai.configure(api_key=userdata.get('GOOGLE_API_KEY'))

def generate_response_and_update_df_gemini(df, row, model, system_prompt, user_prompt, temperature):
    # Configurar el modelo de Gemini
    model_instance = genai.GenerativeModel(model)

    # Combinar system prompt y user prompt para Gemini
    combined_prompt = f"Instrucciones del sistema: {system_prompt}\n\nConsulta del usuario: {user_prompt}"

    # Configurar parámetros de generación
    generation_config = genai.types.GenerationConfig(
        temperature=temperature,
        max_output_tokens=2048,
    )

    try:
        # Llamada a la API de Gemini
        response = model_instance.generate_content(
            combined_prompt,
            generation_config=generation_config
        )

        # Extraer información de la respuesta
        content = response.text

        # Imprimir información (Gemini no proporciona usage como OpenAI)
        print(f"Respuesta generada para fila {row}")
        print(content)

        # Añadir la respuesta al DataFrame
        df.loc[row, 'GPTId'] = row
        df.loc[row, 'GPTContent'] = content
        df.loc[row, 'GPTResponse'] = str(response)
        df.loc[row, 'GPTUsage'] = "No disponible en Gemini"
        df.loc[row, 'PromtAgregado'] = user_prompt

    except Exception as e:
        print(f"Error en fila {row}: {e}")
        df.loc[row, 'GPTId'] = row
        df.loc[row, 'GPTContent'] = f"Error: {e}"
        df.loc[row, 'GPTResponse'] = "Error"
        df.loc[row, 'GPTUsage'] = "Error"
        df.loc[row, 'PromtAgregado'] = user_prompt

    return df

# Llamada a la función en un bucle para Gemini
start_row = 0
end_row = 193
fixed_delay = 5  # Gemini tiene límites diferentes, ajusta según necesidad

# Especifica el modelo de Gemini que quieres usar ver https://ai.google.dev/gemini-api/docs/models?hl=es-419  limites de cada modelo https://ai.google.dev/gemini-api/docs/rate-limits?hl=es-419
ModelUsed = "gemini-2.5-flash-lite-preview-06-17"  # o "gemini-2.5-pro" "gemini-2.5-flash-lite-preview-06-17" "gemini-2.0-flash-lite"

for row in range(start_row, end_row):
    PromtUser1 = DataPapers['Prompt'].iloc[row]
    df3 = generate_response_and_update_df_gemini(df3, row, ModelUsed, PromtSystemB, PromtUser1, Temperatura)
    print(f"Procesada fila {row + 1} de {end_row}")

    # Guardar periódicamente para no perder progreso
    if row % 50 == 0 and row > 0:
        df3.to_csv(f"resultados_parciales_gemini_hasta_fila_{row}.csv", index=False)
        print(f"Progreso guardado hasta la fila {row}")

    # Pausa fija entre peticiones
    if row < end_row - 1:
        time.sleep(fixed_delay)

# Guardar resultado final
df3.to_csv("resultados_finales_gemini.csv", index=False)
print("Proceso completado con Gemini!")

# Codigo especifico para ANTHROPIC/Claude API

In [ ]:
# Para Claude
!pip install anthropic

import time
from anthropic import Anthropic
from google.colab import userdata

# Configurar el cliente de Claude
client_claude = Anthropic(api_key=userdata.get('ClaudeAPIkei'))

def generate_response_and_update_df_claude(df, row, model, system_prompt, user_prompt, temperature):
    try:
        # Llamada a la API de Claude
        response = client_claude.messages.create(
            model=model,
            max_tokens=2048,
            temperature=temperature,
            system=system_prompt,
            messages=[
                {"role": "user", "content": user_prompt}
            ]
        )

        # Extraer información de la respuesta
        content = response.content[0].text
        usage_info = f"Input tokens: {response.usage.input_tokens}, Output tokens: {response.usage.output_tokens}"

        # Imprimir información
        print(f"Tokens usados: {usage_info}")
        print(content)

        # Añadir la respuesta al DataFrame
        df.loc[row, 'GPTId'] = row
        df.loc[row, 'GPTContent'] = content
        df.loc[row, 'GPTResponse'] = str(response)
        df.loc[row, 'GPTUsage'] = usage_info
        df.loc[row, 'PromtAgregado'] = user_prompt

    except Exception as e:
        print(f"Error en fila {row}: {e}")
        df.loc[row, 'GPTId'] = row
        df.loc[row, 'GPTContent'] = f"Error: {e}"
        df.loc[row, 'GPTResponse'] = "Error"
        df.loc[row, 'GPTUsage'] = "Error"
        df.loc[row, 'PromtAgregado'] = user_prompt

    return df

# Llamada a la función en un bucle para Claude
start_row = 0
end_row = 50
fixed_delay = 12  # Claude tiene límites de rate, ajusta según tu plan

# Especifica el modelo de Claude que quieres usar ver info en https://docs.anthropic.com/en/docs/about-claude/models/overview
ModelUsed = "claude-sonnet-4-20250514"  # o "claude-opus-4-20250514" "claude-3-5-haiku-20241022"

for row in range(start_row, end_row):
    PromtUser1 = DataPapers['Prompt'].iloc[row]
    df3 = generate_response_and_update_df_claude(df3, row, ModelUsed, PromtSystemB, PromtUser1, Temperatura)
    print(f"Procesada fila {row + 1} de {end_row}")

    # Guardar periódicamente para no perder progreso
    if row % 50 == 0 and row > 0:
        df3.to_csv(f"resultados_parciales_claude_hasta_fila_{row}.csv", index=False)
        print(f"Progreso guardado hasta la fila {row}")

    # Pausa fija entre peticiones
    if row < end_row - 1:
        time.sleep(fixed_delay)

# Guardar resultado final
df3.to_csv("resultados_finales_claude.csv", index=False)
print("Proceso completado con Claude!")

# Codigo para POE API (todos los modelos)

## celda a ejecutar al principio

In [ ]:
# Versin POE Instalar dependencia
!pip install openai

import os
from openai import OpenAI
import time
import pandas as pd
from google.colab import userdata



# APIKEY de secrets de Colab
API_KEY = userdata.get('KeyPOE')

# Verificar API key
if not API_KEY:
    raise ValueError("No se encontró la API key 'KeyPOE' en los secrets")

# Inicialización del cliente
client = OpenAI(
    api_key=API_KEY,
    base_url="https://api.poe.com/v1",
)





## codigo para iterar los modelos uno a uno manualmente y descarga automaticmente los ficheros excel finales (ya no lo uso, ver modelo iterativo mas adelante)

In [ ]:
# Modelos válidos para Poe (verifica la documentación actual)
ModelUsed="Gemini-1.5-Pro"  # revisar que no tenga una , al final
test_models = [
    "Claude-Sonnet-3.5",  # 190 p
    "Claude-Haiku-4.5",   # 200 p
    "Claude-Sonnet-3.7",  # 800 p
    "Claude-Opus-4.1",    # 5100 p
    "Claude-Sonnet-4.5",    # 870 p
    "Claude-Sonnet-4-Reasoning",  # 1340 p
    "Claude-Opus-4-Reasoning",    # 5100 p
    "GPT-4-Classic",     # 1600 OpenAI's GPT-4 model. Powered by gpt-4-0613
    "gpt-3.5-turbo",   # 25 p
    "GPT-4-Turbo",     # 606 p
    "GPT-5",           # 280 p
    "GPT-5-mini",      #  33 p
    "GPT-5-nano",      #  8 p
    "GPT-4.1",         #  280 p
    "GPT-4.1-mini",    # 34 p
    "Gemini-2.5-Pro",  #  740 p
    "Gemini-2.5-Flash",# 37 p
    "Grok-3",          # 901 p xAI's February 2025 flagship
    "Grok-4",          # 1100 p
    "Grok-3-Mini",     #  50 p
    "DeepSeek-V3.1",   # 260 p
    "Llama-4-Maverick",#  50 p
    "Llama-3.1-70B",   # 39 p
    "Llama-3.1-405B",   # 150 p
    "Llama-3.3-70B-FW", # 150 p
    "Mixtral8x22b-Inst-FW",  # 130 p
    "Mixtral-8x7B-T",   #120 p
    "GLM-4.5-FW",         # 180 p
    "Qwen3-235B-2507-FW", # 90 p
    "Qwen3-32B-CS", # 120 p
    "Qwen3-Next-Instruct-T", # 80 p
    "Qwen3-Next-Think-T", # 100 p
    "Qwen-2.5-7B-T",      # 75 p
    "Qwen-3-235B-2507-T", # 63 p
    "DeepSeek-V3.1", # 260 p
    "DeepSeek-R1-FW",     # 600
    "Deepseek-V3-FW"    #300 p
]

def generate_response_and_update_df(df, row, model, system_prompt, user_prompt, temperature, max_retries=3):
    """
    Función mejorada con manejo de errores y reintentos
    """
    for attempt in range(max_retries):
        try:
            # Llamada a la API
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature
            )

            # Imprimir información
            print(f"✅ Fila {row} procesada exitosamente")
            print(f"Tokens usados: {response.usage}")
            print(response.choices[0].message.content)

            # Actualizar DataFrame
            df.loc[row, 'GPTId'] = row
            df.loc[row, 'GPTContent'] = response.choices[0].message.content
            df.loc[row, 'GPTResponse'] = str(response)
            df.loc[row, 'GPTUsage'] = str(response.usage)
            df.loc[row, 'PromtAgregado'] = user_prompt

            return df

        except Exception as e:
            print(f"❌ Error en fila {row}, intento {attempt + 1}/{max_retries}: {e}")

            if attempt < max_retries - 1:
                # Esperar más tiempo antes del siguiente intento
                wait_time = (attempt + 1) * 2  # 2, 4, 6 segundos
                print(f"⏳ Esperando {wait_time} segundos antes del siguiente intento...")
                time.sleep(wait_time)
            else:
                # Si todos los intentos fallan, guardar error en el DataFrame
                print(f"💥 Todos los intentos fallaron para la fila {row}")
                df.loc[row, 'GPTId'] = row
                df.loc[row, 'GPTContent'] = f"ERROR: {str(e)}"
                df.loc[row, 'GPTResponse'] = "ERROR"
                df.loc[row, 'GPTUsage'] = "ERROR"
                df.loc[row, 'PromtAgregado'] = user_prompt
                return df

# Parámetros del bucle
start_row = 0
end_row = 593  # como hay gesiton de errores igual no es necesario que sea exactamente el numero de filas (porque este no se ejecuta, pero al empezar en 0 la fila anterior es el total). si es mas el proceso se corta y no ejecutar la descarga de ficheros
fixed_delay = 0.05  #evitar el rate limit la idea es no pasarse de 200 peticiones por minuto, incluso sin delay va a unas 30 peticiones por minuto

print(f"🚀 Iniciando procesamiento de filas {start_row} a {end_row-1}")

for row in range(start_row, end_row):
    try:
        PromtUser1 = DataPapers['Prompt'].iloc[row]
        df3 = generate_response_and_update_df(df3, row, ModelUsed, PromtSystemB, PromtUser1, Temperatura)
        print(f"📝 Procesada fila {row + 1} de {end_row}")

        # Guardar periódicamente
        if (row + 1) % 50 == 0:  # Guardar cada 50 filas
            filename = f"resultados_parciales_hasta_fila_{row}.csv"
            df3.to_csv(filename, index=False)
            print(f"💾 Progreso guardado: {filename}")

        # Pausa entre peticiones
        if row < end_row - 1:
            time.sleep(fixed_delay)

    except KeyboardInterrupt:
        print("🛑 Proceso interrumpido por el usuario")
        break
    except Exception as e:
        print(f"💥 Error inesperado en fila {row}: {e}")
        continue

# Guardar resultado final
try:
    df3.to_csv("resultados_finales.csv", index=False)
    print("✅ Proceso completado con POE!")
except Exception as e:
    print(f"❌ Error al guardar archivo final: {e}")

print("📊 Resumen final:")
print(df3.head())


# Guardamos el DataFrame en un archivo Excel
from google.colab import files

# Definimos la ruta y el nombre del archivo incluyendo la fecha actual
from datetime import datetime
fecha_actual = datetime.now().strftime("%Y-%m-%d %H:%M")
nombre_base = f"ART-749-SCAg-api-poe-{ModelUsed}-B01-{Temperatura}"
# nombre_base = f"ART-749-SCAg-api-OpenAi-{ModelUsed}-B01-{Temperatura}"
nombre_fichero = f"{nombre_base}_{fecha_actual}.xlsx"

# Guardamos el archivo
df3.to_excel(nombre_fichero, index=False)

# Verificamos que el archivo existe y lo descargamos
if os.path.exists(nombre_fichero):
    files.download(nombre_fichero)
    print(f"Archivo {nombre_fichero} descargado exitosamente")
else:
    print("Error: No se pudo crear el archivo")

## verificaciones de variables  modelos

In [ ]:
# Verificar que todas las variables estén definidas
required_vars = ['df3', 'DataPapers', 'PromtSystemB', 'Temperatura']
for var_name in required_vars:
    if var_name not in globals():
        print(f"❌ Variable '{var_name}' no está definida")

In [ ]:
# Función para validar el modelo antes de usar:
def validate_model(model_name):
    """Validar si un modelo específico funciona"""
    try:
        # Hacer una petición de prueba muy simple
        test_response = client.chat.completions.create(
            model=model_name,
            messages=[
                    {"role": "system", "content": PromtSystemB},
                    {"role": "user", "content": "23JSJZRE--10.17270/J.LOG.2022.735# ANALYZING NEW WAYS TO ADAPT THE TRIPLE-A SUPPLY CHAIN MODEL AND ITS EXTENSIONS IN AGRI-FOOD SUPPLY CHAINS. Background: With the emergence of supply chain management as a key strategic function in the agri-food sector, a lot of research has been conducted to find ways to improve the performance and sustainability of agri-food supply chains. The Triple-A Supply Chain concept, which refers to the agility, adaptability, and alignment of the supply chains, has been a field of study for various researchers aiming at shaping meaningful and sustainable competitive advantages for businesses and organizations in various sectors. Over the years, alternative, complementary, or upgraded versions of this approach have been proposed, such as the “New AAA Supply Chain”, whi ch describes the renewed Triple-A Supply Chain model based on Super-Agility, Architectural Adaptability, and Ecosystem Alignment, and the “Triple A & R” framework, which refers to Agility for Robustness, Adaptability, and Resilience, and Re-Alignment. Methods: This paper presents the results of a selective study of the bibliography considering the Triple-A Supply Chain model, the “New AAA Supply Chain” model and the “Triple A & R” framework. These frameworks are analyzed and compared with each other considering their principles, and their implementation in the agri-food sector is researched. The scope of this study is to analyze the potential of the application and suitability of these frameworks in agri-food supply chains, having considered the particularities of the sector. Results: Examining the models concerning the evolution of the Triple-A Supply Chain paradigm, it is evident that they differ from each other, as they approach supply chain management from different viewpoints. Conclusions: The potential of application of various models originating from the Triple-A Supply Chain paradigm was examined in the case of the agri-food sector considering product nature, sustainability, and investment cost as the factors affecting it. These frameworks could partially find application in the agri-food sector, as some of their guidelines promote the increase of the agri-food supply chain effectiveness. © Wyższa Szkoła Logistyki, Poznań, Polska."}
                ]

        )
        print(f"✅ Modelo '{model_name}' es válido")
        return True
    except Exception as e:
        print(f"❌ Modelo '{model_name}' no es válido: {e}")
        return False

# Probar diferentes nombres de modelo
test_models = [
    "Claude-Sonnet-3.5",  # 190 p
    "Claude-Haiku-4.5",   # 200 p
    "Claude-Sonnet-3.7",  # 800 p
    "Claude-Opus-4.1",    # 5100 p
    "Claude-Sonnet-4.5",    # 870 p
    "Claude-Sonnet-4-Reasoning",  # 1340 p
    "Claude-Opus-4-Reasoning",    # 5100 p
    "GPT-4-Classic",     # 1600 OpenAI's GPT-4 model. Powered by gpt-4-0613
    "gpt-3.5-turbo",   # 25 p
    "GPT-4-Turbo",     # 606 p
    "GPT-5",           # 280 p
    "GPT-5-mini",      #  33 p
    "GPT-5-nano",      #  8 p
    "GPT-4.1",         #  280 p
    "GPT-4.1-mini",    # 34 p
    "Gemini-2.5-Pro",  #  740 p
    "Gemini-2.5-Flash",# 37 p
    "Grok-3",          # 901 p xAI's February 2025 flagship
    "Grok-4",          # 1100 p
    "Grok-3-Mini",     #  50 p
    "DeepSeek-V3.1",   # 260 p
    "Llama-4-Maverick",#  50 p
    "Llama-3.1-70B",   # 39 p
    "Llama-3.1-405B",   # 150 p
    "Llama-3.3-70B-FW", # 150 p
    "Mixtral8x22b-Inst-FW",  # 130 p
    "Mixtral-8x7B-T",   #120 p
    "GLM-4.5-FW",         # 180 p
    "Qwen3-235B-2507-FW", # 90 p
    "Qwen3-32B-CS", # 120 p
    "Qwen3-Next-Instruct-T", # 80 p
    "Qwen3-Next-Think-T", # 100 p
    "Qwen-2.5-7B-T",      # 75 p
    "Qwen-3-235B-2507-T", # 63 p
    "DeepSeek-V3.1", # 260 p
    "DeepSeek-R1-FW",     # 600
    "Deepseek-V3-FW"    #300 p
]

print("🔍 Probando modelos...")
valid_models = []
for model in test_models:
    if validate_model(model):
        valid_models.append(model)
    time.sleep(1)  # Pausa entre pruebas

print(f"\n✅ Modelos válidos encontrados: {valid_models}")

## Codigo mejorado en POE para iterar los modelos automaticamente

In [ ]:
import time
import os
from datetime import datetime
from google.colab import files
import pandas as pd


def generate_response_and_update_df(df, row, model, system_prompt, user_prompt, temperature, max_retries=3):
    """
    Función mejorada con manejo de errores y reintentos
    """
    for attempt in range(max_retries):
        try:
            # Llamada a la API
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt}
                ],
                temperature=temperature
            )

            # Imprimir información
            print(f"✅ Fila {row} procesada exitosamente")
            print(f"Tokens usados: {response.usage}")
            print(response.choices[0].message.content)

            # Actualizar DataFrame
            df.loc[row, 'GPTId'] = row
            df.loc[row, 'GPTContent'] = response.choices[0].message.content
            df.loc[row, 'GPTResponse'] = str(response)
            df.loc[row, 'GPTUsage'] = str(response.usage)
            df.loc[row, 'PromtAgregado'] = user_prompt

            return df

        except Exception as e:
            print(f"❌ Error en fila {row}, intento {attempt + 1}/{max_retries}: {e}")

            if attempt < max_retries - 1:
                wait_time = (attempt + 1) * 2
                print(f"⏳ Esperando {wait_time} segundos antes del siguiente intento...")
                time.sleep(wait_time)
            else:
                print(f"💥 Todos los intentos fallaron para la fila {row}")
                df.loc[row, 'GPTId'] = row
                df.loc[row, 'GPTContent'] = f"ERROR: {str(e)}"
                df.loc[row, 'GPTResponse'] = "ERROR"
                df.loc[row, 'GPTUsage'] = "ERROR"
                df.loc[row, 'PromtAgregado'] = user_prompt
                return df

def descargar_archivo_inmediato(nombre_fichero, pausa_post=8):
    """
    Descarga inmediata y espera antes de continuar
    """
    try:
        print(f"📥 Descargando: {nombre_fichero}")
        files.download(nombre_fichero)
        print(f"⏳ Esperando {pausa_post}s para asegurar descarga...")
        time.sleep(pausa_post)  # Pausa LARGA para que se complete la descarga
        print(f"✅ Descarga completada: {nombre_fichero}")
    except Exception as e:
        print(f"❌ Error descargando {nombre_fichero}: {e}")


# Variable global para controlar el tiempo de última descarga
ultima_descarga = 0


# =============================================================================
# CONFIGURACIÓN PRINCIPAL
# =============================================================================

# Lista de modelos a procesar
modelos_a_procesar = [
    # "Claude-Sonnet-3.5",  # 190 p
    # "Claude-Haiku-4.5",   # 200 p
    # "Claude-Sonnet-3.7",  # 800 p
    # "Claude-Opus-4.1",    # 5100 p
    # "Claude-Sonnet-4.5",    # 870 p
    # "Claude-Sonnet-4-Reasoning",  # 1340 p
    # "Claude-Opus-4-Reasoning",    # 5100 p
    # "DeepSeek-V3.1", # 260 p
    "DeepSeek-R1-FW",     # 600
    "Deepseek-V3-FW",   #300 p
    # "GPT-4-Classic",     # 1600 OpenAI's GPT-4 model. Powered by gpt-4-0613
    # "gpt-3.5-turbo",   # 25 p
    # "GPT-4-Turbo",     # 606 p
    # "GPT-5",           # 280 p
    # "GPT-5-mini",      #  33 p
    # "GPT-5-nano",      #  8 p
    # "GPT-4.1",         #  280 p
    # "GPT-4.1-mini",    # 34 p
    # "Gemini-2.5-Pro",  #  740 p
    # "Gemini-2.5-Flash",# 37 p
    # "Grok-3",          # 901 p xAI's February 2025 flagship
    # "Grok-4",          # 1100 p
    # "Grok-3-Mini",     #  50 p
    # "Llama-4-Maverick",#  50 p
    # "Llama-3.1-70B",   # 39 p
    # "Llama-3.1-405B",   # 150 p
    # "Llama-3.3-70B-FW", # 150 p
    # "Mixtral8x22b-Inst-FW",  # 130 p
    # "Mixtral-8x7B-T",   #120 p
    # "GLM-4.5-FW",         # 180 p
    # "Qwen3-235B-2507-FW", # 90 p
    # "Qwen3-32B-CS", # 120 p
    # "Qwen3-Next-Instruct-T", # 80 p
    # "Qwen3-Next-Think-T", # 100 p
    # "Qwen-2.5-7B-T",      # 75 p
    # "Qwen-3-235B-2507-T" # 63 p
]

# # solo para pruebas, comentar cuando uso en producción
# modelos_a_procesar = [
#     "GPT-5-nano"      #  8 p
# ]

# Parámetros generales
start_row = 19
end_row = 20  # aunuqe empiece en cero, la ultima linea es igual al numero de registros.. si hay 20 filas, endrow es 20, no 19
fixed_delay = 0.05
nombre_base_archivo = "ART-749-SCAg-api-poe-Pilot20"
nombre_base_archivo = "prueba" #comentar cuando no este probando

# #  la "temperatura" es un hiperparámetro que controla la aleatoriedad de las predicciones del modelo.
#  Un valor de temperatura más alto (como 0.8 o 1.0) hará que el modelo haga predicciones más diversas y creativas,
#  mientras que un valor más bajo (como 0.2 o 0.3) hará que las predicciones sean más concentradas y deterministas.
# # Dado que nuestra tarea implica la extracción de entidades, la clasificación y la explicación,
# querría que las respuestas del modelo sean coherentes y directas en lugar de ser demasiado creativas.
# Por lo tanto, sería recomendable una temperatura  baja, posiblemente alrededor de 0.3.

Temperatura= 1 #por encima de 0.5 pocas veces sigue las instrucciones (no llega al cuarto paso o no pone los # donde toca)
#algunos modelos de OpenAI no permiten trabajar con temperatura diferente de 1 cuando usas API de OpenAI, pero no hay problema con la de POE

print(f"\n{'#'*80}")
print(f"🚀 INICIO DEL PROCESAMIENTO AUTOMÁTICO")
print(f"📋 Total de modelos a procesar: {len(modelos_a_procesar)}")
print(f"📊 Filas a procesar por modelo: {end_row - start_row}")
print(f"{'#'*80}\n")

# =============================================================================
# BUCLE PRINCIPAL - PROCESAR TODOS LOS MODELOS
# =============================================================================

for idx, ModelUsed in enumerate(modelos_a_procesar, 1):
    print(f"\n{'='*80}")
    print(f"🎯 MODELO {idx}/{len(modelos_a_procesar)}: {ModelUsed}")
    print(f"{'='*80}\n")

    # CREAR df3 NUEVO PARA CADA MODELO
    df3 = DataPapers.copy()
    df3['GPTId'] = None
    df3['GPTContent'] = None
    df3['GPTResponse'] = None
    df3['GPTUsage'] = None
    df3['PromtAgregado'] = None

    try:
        print(f"🚀 Iniciando procesamiento de filas {start_row} a {end_row-1}")

        for row in range(start_row, end_row):
            try:
                PromtUser1 = DataPapers['Prompt'].iloc[row]
                df3 = generate_response_and_update_df(df3, row, ModelUsed, PromtSystemB, PromtUser1, Temperatura)
                print(f"📝 Procesada fila {row + 1} de {end_row}")

                # Guardar periódicamente
                if (row + 1) % 50 == 0:
                    filename = f"resultados_parciales_hasta_fila_{row}.csv"
                    df3.to_csv(filename, index=False)
                    print(f"💾 Progreso guardado: {filename}")

                # Pausa entre peticiones
                if row < end_row - 1:
                    time.sleep(fixed_delay)

            except KeyboardInterrupt:
                print("🛑 Proceso interrumpido por el usuario")
                break
            except Exception as e:
                print(f"💥 Error inesperado en fila {row}: {e}")
                continue

        # Guardar resultado final
        try:
            df3.to_csv("resultados_finales.csv", index=False)
            print("✅ Proceso completado con POE!")
        except Exception as e:
            print(f"❌ Error al guardar archivo final: {e}")

        print("📊 Resumen final:")
        print(df3.head())

        # Guardamos el DataFrame en un archivo Excel
        fecha_actual = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
        nombre_base = f"{nombre_base_archivo}-{ModelUsed}-{Temperatura}"
        nombre_fichero = f"{nombre_base}_{fecha_actual}.xlsx"

        df3.to_excel(nombre_fichero, index=False)

        if os.path.exists(nombre_fichero):
          descargar_archivo_inmediato(nombre_fichero, pausa_post=8)
        else:
          print("❌ Error: No se pudo crear el archivo")

        print(f"\n✅ MODELO {ModelUsed} COMPLETADO\n")

    except Exception as e:
        print(f"\n❌ ERROR CRÍTICO procesando modelo {ModelUsed}: {e}")
        print(f"⏭️ Continuando con el siguiente modelo...\n")
        continue

print(f"\n{'#'*80}")
print(f"🎉 TODOS LOS MODELOS HAN SIDO PROCESADOS")
print(f"{'#'*80}\n")

# codigo comun para guardar los resultados

In [ ]:
# Guardamos el DataFrame en un archivo Excel
from google.colab import files

# Definimos la ruta y el nombre del archivo incluyendo la fecha actual
from datetime import datetime
fecha_actual = datetime.now().strftime("%Y-%m-%d %H:%M")
nombre_base = f"ART-749-SCAg-api-poe-{ModelUsed}-B01-{Temperatura}"
# nombre_base = f"ART-749-SCAg-api-OpenAi-{ModelUsed}-B01-{Temperatura}"
nombre_fichero = f"{nombre_base}_{fecha_actual}.xlsx"

# Guardamos el archivo
df3.to_excel(nombre_fichero, index=False)

# Verificamos que el archivo existe y lo descargamos
if os.path.exists(nombre_fichero):
    files.download(nombre_fichero)
    print(f"Archivo {nombre_fichero} descargado exitosamente")
else:
    print("Error: No se pudo crear el archivo")

# mejoras



* crear una fiuncion para cada Proeedero y usar un unico codigo que dependinde del proveedor (gmini, openai, claude, o poe9) coge la funciona que toca, pero el resto de codigo es el mismo (no como ahora que lo tengo cuadruplicado)